# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join, exists
import joblib
import sys
import torch
from os import rename
sys.path.append("../../")

from src.configs.default_configs import fn_model, fn_pred, fn_pred_perf, fn_ue_perf
from src.configs.lung_config import data_name, postnet_param
from src.file_manager.filepath import FilePath
from src.evaluation.misc import combine_pred_df
from model_ue_dict import pred_model_files, perf_model_files, ue_dict
from src.evaluation.evaluate_old import evaluate_ue
from seed_file import seed
# seed = 2024

batch_size = 32
eval_batch_size = 128

tuning_seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_data_file = join(fp.get_preprocessed_folder(), "raw_data", "lungcancerdataset.csv")
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

directory_model=fp.get_parent_folder(folder_name=fn_model)
directory_results=fp.get_parent_folder(folder_name=fn_pred)
fn_cur_model = f"model-dpn-{seed}-{data_name}-{seed}-{postnet_param['architecture']}-{postnet_param['input_dims']}-{postnet_param['output_dim']}"

# Load Data

In [ ]:
split_dict_scaled = joblib.load(fp_split_dict_file)
feat_cols_w_pc = ['pc1', 'pc2', 'pc3', 'age_interview', 'BMI', 'telomere length', 'Leisure screen time', 'ahei2010score', 'amed', 'dash', 'SBP', 'DBP', 'pgs000070', 'pgs000721', 'sex (0=Male, 1=Female)', 'alcohol_0_12', 'smoke_ex(1)', 'smoke_current(2)', 'prevalent diabetes']
target_col = "lung cancer"

# Load Predictions

In [ ]:
# load all predictions
test_len = len(split_dict_scaled["test_df"])
all_pred_dfs = []
for model_file in pred_model_files:
    fp_predictions_file = join(fp.get_parent_folder(fn_pred), model_file+".csv")
    pred_df = pd.read_csv(fp_predictions_file, index_col=0).iloc[-test_len:]
    all_pred_dfs.append(pred_df)
pred_df = combine_pred_df(all_pred_dfs) 

# Prediction Performance

In [ ]:
all_perf_df = []
for perf_file in perf_model_files:
    fp_perf_file = join(fp.get_parent_folder("perf_evaluation"), perf_file+".csv")
    perf_df = pd.read_csv(fp_perf_file, index_col=0).loc[["Test"]]
    perf_df.index = [perf_file]
    all_perf_df.append(perf_df)
all_perf_df = pd.concat(all_perf_df)
all_perf_df.to_csv(join(fp.get_parent_folder(fn_ue_perf), "pred_perf.csv"))
display(all_perf_df)

# Evaluate UE

In [ ]:
evaluate_ue(
    pred_df=pred_df, stat_dict=ue_dict, target_col=target_col, 
    fp_evaluation=fp.get_parent_folder(fn_ue_perf)
)

In [ ]:
all_ue_perf_df = []
for ue_file in ue_dict.keys():
    fp_ue_perf_df = join(fp.get_parent_folder(fn_ue_perf), ue_file+".csv")
    ue_perf_df = pd.read_csv(fp_ue_perf_df, index_col=0)
    ue_perf_df.index = [ue_file]
    all_ue_perf_df.append(ue_perf_df)
all_ue_perf_df = pd.concat(all_ue_perf_df)
all_ue_perf_df
all_ue_perf_df.to_csv(join(fp.get_parent_folder(fn_ue_perf), "ue_perf.csv"))
display(all_ue_perf_df)